# Scrollie — MuscleMap vs MuscleMap + MedSAM (mask prompt)

Pick a stack and scroll through slices comparing:
- **Left**: original fat-fraction image
- **Centre**: MuscleMap WB segmentation overlay
- **Right**: MuscleMap + MedSAM mask-prompt overlay (if available)

In [ ]:
import glob
import os
import re
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import SimpleITK as sitk
from ipywidgets import IntSlider, Dropdown, VBox
import ipywidgets as widgets
from IPython.display import display

In [ ]:
MM_SEGS_DIR      = "MuscleMap_segs"
MM_MASKPROMPT_DIR = "MuscleMap_WB_maskprompt"
IMAGE_BASE       = "myosegmenTUM"

LABEL_MAP = {
    7101: "Vastus_Lateralis_L",
    7102: "Vastus_Lateralis_R",
    7111: "Vastus_Intermedius_L",
    7112: "Vastus_Intermedius_R",
    7121: "Vastus_Medialis_L",
    7122: "Vastus_Medialis_R",
    7131: "Rectus_Femoris_L",
    7132: "Rectus_Femoris_R",
    7141: "Sartorius_L",
    7142: "Sartorius_R",
    7151: "Gracilis_L",
    7152: "Gracilis_R",
    7161: "Semimembranosus_L",
    7162: "Semimembranosus_R",
    7171: "Semitendinosus_L",
    7172: "Semitendinosus_R",
    7181: "Biceps_Femoris_L",
    7182: "Biceps_Femoris_R",
    7201: "Adductor_Magnus_L",
    7202: "Adductor_Magnus_R",
}

seg_files = sorted(glob.glob(os.path.join(MM_SEGS_DIR, "*_dseg.nii.gz")))

def seg_to_nii(seg_path):
    stem    = os.path.basename(seg_path).replace("_dseg.nii.gz", "")
    subject = stem.split("_FATFRACTION")[0]
    m       = re.search(r"stack(\d+)", stem)
    stack_n = m.group(1)
    return os.path.join(IMAGE_BASE, subject, "ImageData",
                        f"{subject}_FATFRACTION",
                        f"{subject}_FATFRACTION_stack{stack_n}.nii")

file_options = {os.path.basename(p).replace("_dseg.nii.gz", ""): p for p in seg_files}
print(f"Found {len(file_options)} segmented stacks")

In [ ]:
def build_overlay(segmentation, label_map, cmap):
    overlay = np.zeros((*segmentation.shape, 4), dtype=float)
    for i, (label_idx, name) in enumerate(label_map.items()):
        color = cmap(i)
        overlay[segmentation == label_idx] = [color[0], color[1], color[2], 0.5]
    return overlay

def load_mm_stack(seg_path):
    nii_path  = seg_to_nii(seg_path)
    img_sitk  = sitk.ReadImage(nii_path)
    img_array = sitk.GetArrayFromImage(img_sitk).astype(float)
    gt_norm   = (img_array - img_array.min()) / (img_array.max() - img_array.min() + 1e-8)

    seg_sitk  = sitk.ReadImage(seg_path)
    seg_array = sitk.GetArrayFromImage(seg_sitk)

    present     = {k: v for k, v in LABEL_MAP.items() if np.any(seg_array == k)}
    label_map   = {i: name for i, (_, name) in enumerate(present.items(), start=1)}
    name_to_seq = {name: i for i, name in label_map.items()}

    seg_seq = np.zeros_like(seg_array, dtype=np.uint16)
    for orig_k, name in present.items():
        seg_seq[seg_array == orig_k] = name_to_seq[name]

    cmap    = plt.colormaps["tab20"].resampled(max(len(label_map), 1))
    overlay = build_overlay(seg_seq, label_map, cmap)
    return gt_norm, overlay, label_map, cmap, name_to_seq

def load_npz_overlay(npz_path, name_to_seq, label_map, shape, cmap):
    data    = np.load(npz_path)
    seg_seq = np.zeros(shape, dtype=np.uint16)
    for name, arr in data.items():
        seq_idx = name_to_seq.get(name)
        if seq_idx is not None:
            seg_seq[arr > 0] = seq_idx
    return build_overlay(seg_seq, label_map, cmap)

In [ ]:
file_dropdown = Dropdown(options=list(file_options.keys()), description="Stack:")
slice_slider  = IntSlider(min=0, max=1, step=1, value=0, description="Slice:",
                           layout=widgets.Layout(width="600px"))
out = widgets.Output()

_cache = {}

def get_data(label):
    if label not in _cache:
        seg_path = file_options[label]
        gt_norm, overlay_mm, label_map, cmap, name_to_seq = load_mm_stack(seg_path)

        npz_path = os.path.join(MM_MASKPROMPT_DIR, f"{label}_mm_maskprompt.npz")
        if os.path.exists(npz_path):
            overlay_refined = load_npz_overlay(
                npz_path, name_to_seq, label_map, gt_norm.shape, cmap
            )
        else:
            overlay_refined = None

        legend_patches = [
            mpatches.Patch(color=cmap(i), alpha=0.6, label=name)
            for i, (_, name) in enumerate(label_map.items())
        ]
        _cache[label] = (gt_norm, overlay_mm, overlay_refined, legend_patches)
        slice_slider.max = gt_norm.shape[0] - 1
        slice_slider.value = 0
    return _cache[label]

def on_file_change(change):
    _cache.clear()
    get_data(change["new"])
    render(file_dropdown.value, slice_slider.value)

def render(label, slice_idx):
    gt_norm, overlay_mm, overlay_refined, legend_patches = get_data(label)
    n_panels = 3 if overlay_refined is not None else 2
    fig, axes = plt.subplots(1, n_panels, figsize=(6 * n_panels, 6))
    img = gt_norm[slice_idx]

    axes[0].imshow(img, cmap="gray", origin="lower")
    axes[0].set_title(f"Image — slice {slice_idx}")
    axes[0].axis("off")

    axes[1].imshow(img, cmap="gray", origin="lower")
    axes[1].imshow(overlay_mm[slice_idx], origin="lower")
    axes[1].set_title("MuscleMap")
    axes[1].axis("off")
    axes[1].legend(handles=legend_patches, loc="lower right", fontsize=6, framealpha=0.7)

    if overlay_refined is not None:
        axes[2].imshow(img, cmap="gray", origin="lower")
        axes[2].imshow(overlay_refined[slice_idx], origin="lower")
        axes[2].set_title("MuscleMap + MedSAM (mask prompt)")
        axes[2].axis("off")

    fig.suptitle(label, fontsize=10)
    plt.tight_layout()
    with out:
        out.clear_output(wait=True)
        plt.show()

def on_slice_change(change):
    render(file_dropdown.value, change["new"])

file_dropdown.observe(on_file_change, names="value")
slice_slider.observe(on_slice_change, names="value")

if file_options:
    get_data(file_dropdown.value)
    render(file_dropdown.value, 0)

display(VBox([file_dropdown, slice_slider, out]))